# Unidad 3 · Métodos avanzados · Laboratorios en Google Colab

Cuaderno para hacer **todos los laboratorios de la Unidad 3** en Colab Pro+ sin instalar nada en vuestro ordenador.

**Cómo usarlo**
1. Al empezar cada sesión, ejecutad la sección **0 · Preparación** completa (unos 2 minutos).
2. Id a la sección del laboratorio del día. Cada una explica el objetivo, qué lanzar y qué mirar.
3. Los resultados (`logs/`, `optuna/`) se guardan en **Google Drive**: si Colab se desconecta, no se pierden.

**Consejos para Colab Pro+**
- No hace falta GPU: todo va bien en CPU. Usad *Entorno de ejecución → Cambiar tipo → CPU, RAM alta* (más núcleos).
- Los entrenamientos largos se lanzan **en segundo plano** con `lanzar(...)` (sección 0.4). Podéis seguirlos con `estado()` y TensorBoard mientras hacéis otra cosa.
- Con Pro+ la sesión sigue ejecutándose aunque cerréis la pestaña (ejecución en segundo plano).

## 0 · Preparación

### 0.1 · Google Drive y repositorio
Todo vive en `Mi unidad/practica2_rl_snake`, que es un clon del repositorio del grupo (público: clonar no necesita token).

**La primera vez** (cada miembro): en Colab, *Archivo → Abrir cuaderno → GitHub*, abrid `Illera03/practica2_rl_snake` → `colab_unidad3.ipynb` y ejecutad esta celda: clona el repo en vuestro Drive. A partir de ahí abrid siempre el cuaderno desde Drive.

**Para hacer push** hace falta un token de GitHub guardado en los **Secretos** de Colab (icono de la llave) con el nombre `GITHUB_TOKEN` y el acceso desde el cuaderno activado. Sin token la celda funciona igual, pero solo podréis hacer pull.


In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess
REPO = 'github.com/Illera03/practica2_rl_snake.git'
WORK = '/content/drive/MyDrive/practica2_rl_snake'

if not os.path.isdir(f'{WORK}/.git'):
    subprocess.run(['git', 'clone', f'https://{REPO}', WORK], check=True)
os.chdir(WORK)

try:                                   # con token, el remoto queda listo para push
    token = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://{token}@{REPO}'], check=True)
    print('Token encontrado: podéis hacer push.')
except Exception:
    print('Sin GITHUB_TOKEN: solo pull (ved 0.1 para añadirlo).')
print('Carpeta de trabajo:', os.getcwd())


Mounted at /content/drive
Token encontrado: podéis hacer push.
Carpeta de trabajo: /content/drive/MyDrive/practica2_rl_snake


### 0.2 · Sincronizar con el grupo
Traed los cambios de los demás al empezar. Al cerrar cada iteración: `git add`, `git commit` y `git push` (línea comentada de abajo).
Poned vuestro nombre y correo la primera vez.


In [3]:
!git config user.name  "--"      # cambiadlo
!git config user.email "--"  # cambiadlo
!git pull --rebase --autostash
!git status --short
# Al cerrar una iteración:
# !git add DIARIO.md experiments_3 logs/lab_3_3 && git commit -m "it01_original: ..." && git push


error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
 M colab_unidad3.ipynb


### 0.3 · Instalación
Hay que repetirla en cada sesión nueva de Colab (el entorno se reinicia; Drive no).

In [ ]:
!apt-get -qq install -y swig > /dev/null
!pip -q install "stable-baselines3[extra]>=2.4" "gymnasium[box2d]>=1.0" gymnasium-robotics optuna optuna-dashboard sb3-contrib rl_zoo3 moviepy imageio

import os
os.environ['MUJOCO_GL'] = 'egl'        # MuJoCo sin pantalla (Lab 3.6)
import torch, gymnasium, stable_baselines3, mujoco
print('torch', torch.__version__, '| gymnasium', gymnasium.__version__,
      '| sb3', stable_baselines3.__version__, '| mujoco', mujoco.__version__)
!nproc

### 0.4 · Utilidades: lanzar en segundo plano, ver estado, gráficas y GIF
- `ejecutar(cmd)`: lanza y **espera** (para pruebas cortas).
- `lanzar(nombre, cmd, semillas)`: lanza **una semilla por proceso** en segundo plano; la salida va a `logs/procesos/`. Si hay más procesos que núcleos, esperan en cola y arrancan solos.
- `estado()`: qué procesos siguen en marcha y la última línea de cada uno.
- `grafica(...)`: ejecuta `plot_results.py` y muestra la figura.
- `ver_gif(ruta)`: muestra un GIF en el cuaderno.

Si se reinicia el entorno, la cola se pierde: volved a lanzar lo que no hubiera terminado.

In [ ]:
import subprocess, glob, time, threading
from IPython.display import Image, display

os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs('logs/procesos', exist_ok=True)
MAX_PARALELO = os.cpu_count()          # procesos simultáneos; el resto espera en cola
PROCESOS, COLA = {}, []

def _gestor():
    while True:
        activos = sum(p.poll() is None for p, _ in PROCESOS.values())
        while COLA and activos < MAX_PARALELO:
            etiqueta, c = COLA.pop(0)
            log = open(f'logs/procesos/{etiqueta}.log', 'w')
            PROCESOS[etiqueta] = (subprocess.Popen(c, shell=True, stdout=log, stderr=subprocess.STDOUT), c)
            activos += 1
        time.sleep(3)

if not any(t.name == 'gestor_rl' for t in threading.enumerate()):
    threading.Thread(target=_gestor, name='gestor_rl', daemon=True).start()

def ejecutar(cmd):
    # Ejecuta y espera, mostrando la salida.
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-6000:], r.stderr[-3000:])

def lanzar(nombre, cmd, semillas=(None,)):
    # Una semilla por proceso, en segundo plano (con cola). Usad {s} en cmd donde vaya la semilla.
    for s in semillas:
        etiqueta = nombre if s is None else f'{nombre}_s{s}'
        COLA.append((etiqueta, cmd if s is None else cmd.format(s=s)))
        print('en cola:', etiqueta)

def estado():
    for etiqueta, _ in COLA:
        print(f'{etiqueta:34s} en cola')
    for etiqueta, (p, c) in PROCESOS.items():
        rc = p.poll()
        txt = 'en marcha' if rc is None else ('terminado' if rc == 0 else f'ERROR ({rc})')
        try:
            ultima = [l for l in open(f'logs/procesos/{etiqueta}.log').read().splitlines() if l.strip()][-1][:110]
        except (IndexError, FileNotFoundError):
            ultima = ''
        print(f'{etiqueta:34s} {txt:11s} {ultima}')

def esperar():
    while COLA or any(p.poll() is None for p, _ in PROCESOS.values()):
        time.sleep(30)
    estado()

def grafica(*carpetas, metric='reward', configs=None, out=None):
    out = out or f'logs/{os.path.basename(carpetas[0])}_{metric}.png'
    extra = f' --configs {configs}' if configs else ''
    ejecutar(f'python experiments_3/plot_results.py {" ".join(carpetas)} --metric {metric}{extra} --out {out}')
    if os.path.exists(out):
        display(Image(out))

def ver_gif(ruta):
    display(Image(open(ruta, 'rb').read()))

print('Procesos simultáneos:', MAX_PARALELO)

### 0.5 · TensorBoard
Se actualiza solo (botón de recarga arriba a la derecha). Filtrad por laboratorio con la caja de búsqueda (`lab_3_1`, `lab_3_3`…).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tb

---
# Semana 1

## Lab 3.1 · DQN con Stable-Baselines3: de CartPole a LunarLander
**Objetivo.** Primer contacto con SB3 y con un entorno que no se puede resolver con tablas.

**Qué hacer**
1. CartPole con los valores por defecto de SB3 y con los de rl_zoo3. ¿Cuál aprende en 50k pasos?
2. LunarLander-v3 con la configuración de las diapositivas, 3 semillas (en paralelo).
3. Comparar en TensorBoard `eval/mean_reward` frente a `rollout/ep_rew_mean`. ¿Por qué difieren?
4. Grabar un vídeo del mejor modelo.

**Qué deberíais ver.** LunarLander se considera resuelto con unos 200 puntos de media; las semillas no llegan a la vez.

In [ ]:
# 1 · CartPole: valores por defecto frente a rl_zoo3 (unos minutos cada uno)
lanzar('3.1_cartpole_default', 'python experiments_3/lab_3_1_dqn.py --env CartPole-v1 --config default --seeds 0')
lanzar('3.1_cartpole_zoo',     'python experiments_3/lab_3_1_dqn.py --env CartPole-v1 --config zoo --seeds 0')

In [ ]:
# 2 · LunarLander, 3 semillas en paralelo (150k pasos cada una)
lanzar('3.1_lunar', 'python experiments_3/lab_3_1_dqn.py --env LunarLander-v3 --config slides --seeds {s}', semillas=[0, 1, 2])

In [ ]:
estado()

In [ ]:
# 3 · Curvas (cuando terminen)
grafica('logs/lab_3_1')

In [ ]:
# 4 · Vídeo del mejor modelo de la semilla 0
ejecutar('python experiments_3/lab_3_1_dqn.py --env LunarLander-v3 --video logs/lab_3_1/LunarLander-v3_slides/seed_0/best_model.zip')
from IPython.display import Video
videos = sorted(glob.glob('logs/lab_3_1/LunarLander-v3_slides/seed_0/videos/*.mp4'))
Video(videos[-1], embed=True) if videos else print('No hay vídeo (¿falta moviepy?)')

## Lab 3.2 · Ablación: ¿qué pieza de DQN importa más?
**Objetivo.** Entender por qué funciona DQN quitándole piezas, una cada vez.

| Variante | Qué cambia |
|---|---|
| `base` | configuración completa |
| `sin_red_objetivo` | `target_update_interval=1` |
| `buffer_min` | `buffer_size=1_000` |
| `sin_explor_final` | `exploration_final_eps=0.0` |

**Entregable interno.** Una gráfica con las cuatro variantes y tres frases de conclusión, relacionadas con la tríada mortal.

Son 12 entrenamientos (4 variantes × 3 semillas): se lanzan todos y la cola los reparte entre los núcleos.

In [ ]:
for v in ['base', 'sin_red_objetivo', 'buffer_min', 'sin_explor_final']:
    lanzar(f'3.2_{v}', f'python experiments_3/lab_3_2_ablacion.py --variants {v} --seeds {{s}}', semillas=[0, 1, 2])

In [ ]:
estado()

In [ ]:
grafica('logs/lab_3_2')

---
# Semana 2

## Lab 3.3 · Mejorar un agente de REINFORCE en el Snake (implementación evaluable, 15 %)
**Objetivo.** No se evalúa cuántas manzanas come vuestro agente, sino cómo llegáis a él: qué probáis, por qué, qué esperabais que pasara y qué decidís con lo que sale. Las instrucciones completas y la rúbrica están en el `README.md`.

**Ficheros que tenéis que completar** (panel de archivos de la izquierda, carpeta `experiments_3`; los cambios se guardan en Drive):
- `lab_3_3_reinforce.py`: TODO 1–5 (red de política, reward-to-go, partida, pérdida, baseline). Es un requisito: tiene que pasar `--check`.
- `snake_design.py`: TODO 6 (observación) y TODO 7 (recompensa, si decidís cambiarla).
- `DIARIO.md`: una ficha por iteración (hipótesis, cambio, predicción, resultado y decisión), escrita **antes** de lanzar cada experimento.

**Cómo se trabaja.** Cada iteración cambia una sola cosa y se lanza con su nombre (`TAG`). Explorar barato (2 semillas, 100 000 pasos) y confirmar caro (5 semillas, 200 000 pasos) solo las configuraciones finales.

**Orientación (no se puntúa).** Una política aleatoria come 0,04 manzanas por partida; REINFORCE con la observación original, 1–2; con una buena observación, más de 10.

In [ ]:
# Ver el entorno: una partida aleatoria (GIF) y cómo acaban las partidas al azar
ejecutar('python experiments_3/snake_env.py')
import sys, imageio, numpy as np
sys.path.insert(0, 'experiments_3')
import snake_env, gymnasium as gym
env = gym.make('DeustoSnake-v0', render_mode='rgb_array')
frames, (obs, info), done = [], env.reset(seed=0), False
while not done:
    frames.append(env.render())
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    done = term or trunc
print('Esta partida acabó por:', info['fin'])
imageio.mimsave('logs/snake_aleatorio.gif', frames, duration=0.15, loop=0)
ver_gif('logs/snake_aleatorio.gif')

In [ ]:
# Autocomprobaciones: ejecutad tras completar cada TODO
ejecutar('python experiments_3/lab_3_3_reinforce.py --check')
ejecutar('python experiments_3/lab_3_3_reinforce.py --env CartPole-v1 --seeds 0 --steps 50000')   # debe aprender

In [ ]:
# EXPLORACIÓN · una iteración del diario: 2 semillas, 100 000 pasos
# Antes de ejecutar: escribid en DIARIO.md la hipótesis, el cambio y la predicción.
TAG = 'it01_original'                     # nombre de la iteración (itNN_algo_que_la_describa)
OPCIONES = '--obs original --reward sparse'   # el ÚNICO cambio respecto a la iteración anterior
lanzar(TAG, f'python experiments_3/lab_3_3_reinforce.py {OPCIONES} --steps 100000 --tag {TAG} --seeds {{s}}', semillas=range(2))

In [ ]:
# CONFIRMACIÓN · solo las 2–3 configuraciones finales: 5 semillas, 200 000 pasos
TAG = 'it07_final'
OPCIONES = '--obs custom --reward sparse --baseline'
lanzar(TAG, f'python experiments_3/lab_3_3_reinforce.py {OPCIONES} --tag {TAG} --seeds {{s}}', semillas=range(5))

In [ ]:
# Referencias de SB3 con vuestro diseño final (mismo presupuesto, 5 semillas)
TAG = 'it09_ref'
OPCIONES = '--obs custom --reward sparse'
lanzar(TAG + '_ppo', f'python experiments_3/lab_3_3_referencia_sb3.py --algos ppo {OPCIONES} --tag {TAG} --seeds {{s}}', semillas=range(5))
lanzar(TAG + '_a2c', f'python experiments_3/lab_3_3_referencia_sb3.py --algos a2c {OPCIONES} --tag {TAG} --seeds {{s}}', semillas=range(5))

In [ ]:
estado()

In [ ]:
# Tabla del diario: manzanas, intervalo, cómo acaban las partidas y qué cambió en cada iteración.
# Copiadla a DIARIO.md (también queda en logs/tabla_diario.md).
from IPython.display import Markdown
ejecutar('python experiments_3/diario.py logs/lab_3_3 --out logs/tabla_diario.md')
if os.path.exists('logs/tabla_diario.md'):
    display(Markdown(open('logs/tabla_diario.md', encoding='utf-8').read()))

In [ ]:
grafica('logs/lab_3_3', metric='score')          # configs='it07_final,it09_ref_ppo' para elegir curvas

In [ ]:
# Ver jugar a vuestro agente (GIF de una partida) para diagnosticar
TAG, OPCIONES = 'it01_original', '--obs original'
ejecutar(f'python experiments_3/lab_3_3_reinforce.py {OPCIONES} --watch logs/lab_3_3/{TAG}/seed_0/policy.pt --gif logs/snake_{TAG}.gif')
ver_gif(f'logs/snake_{TAG}.gif')

---
# Semana 3

## Lab 3.4 · PPO frente a A2C en LunarLander-v3
**Objetivo.** Comparar dos métodos on-policy entre sí y con el DQN del Lab 3.1 en el mismo entorno.

**Qué hacer**
1. PPO y A2C con 8 entornos en paralelo, 500k pasos, 3 semillas.
2. Vigilar en TensorBoard `train/approx_kl`, `train/clip_fraction`, `train/entropy_loss` y `train/explained_variance`.
3. Barrido de `clip_range`: 0,1 / 0,2 / 0,4.
4. Pregunta: ¿qué algoritmo es más eficiente en muestras? ¿Y en tiempo de reloj?

Cada entrenamiento usa 8 entornos en un mismo proceso: con pocos núcleos, tardará más.

In [ ]:
lanzar('3.4_ppo', 'python experiments_3/lab_3_4_ppo_a2c.py --algos ppo --seeds {s}', semillas=[0, 1, 2])
lanzar('3.4_a2c', 'python experiments_3/lab_3_4_ppo_a2c.py --algos a2c --seeds {s}', semillas=[0, 1, 2])

In [ ]:
# Barrido de clip_range (más tarde, o en otra sesión)
lanzar('3.4_clip', 'python experiments_3/lab_3_4_ppo_a2c.py --algos ppo --clip 0.1,0.2,0.4 --seeds {s}', semillas=[0])

In [ ]:
estado()

In [ ]:
grafica('logs/lab_3_4')
grafica('logs/lab_3_4', 'logs/lab_3_1', configs='ppo_default,a2c_default,LunarLander-v3_slides', out='logs/lunar_comparacion.png')

## Lab 3.5 · Control continuo: Pendulum y la ducha
**Objetivo.** Practicar DDPG, TD3 y SAC, y cerrar el ejercicio de la ducha de la Unidad 1 con acción continua.

**Qué hacer**
1. DDPG, TD3 y SAC en Pendulum-v1 (20k pasos). En torno a −200 o mejor, el péndulo se mantiene arriba.
2. TD3 con ruido de exploración σ = 0 / 0,1 / 0,5. ¿Qué pasa sin ruido?
3. SAC y PPO en la ducha (`ShowerContinuous-v0`). Referencia: un controlador proporcional pasa unos 58 s de 60 dentro del rango; uno aleatorio, unos 9.
4. Ampliación: penalizar cambios bruscos del grifo.

In [ ]:
ejecutar('python experiments_3/shower_continuous.py')     # comprobación y referencias de la ducha
lanzar('3.5_pendulum', 'python experiments_3/lab_3_5_continuo.py --env Pendulum-v1 --algos ddpg,td3,sac --seeds {s}', semillas=[0, 1, 2])
lanzar('3.5_sigma',    'python experiments_3/lab_3_5_continuo.py --env Pendulum-v1 --algos td3 --sigma 0.0,0.1,0.5 --seeds {s}', semillas=[0, 1, 2])

In [ ]:
lanzar('3.5_ducha',    'python experiments_3/lab_3_5_continuo.py --env ShowerContinuous-v0 --algos sac,ppo --steps 30000 --seeds {s}', semillas=[0, 1, 2])
lanzar('3.5_ducha_pen','python experiments_3/lab_3_5_continuo.py --env ShowerContinuous-v0 --algos sac --steps 30000 --change-penalty 0.2 --seeds {s}', semillas=[0, 1, 2])

In [ ]:
estado()

In [ ]:
grafica('logs/lab_3_5_Pendulum-v1')
grafica('logs/lab_3_5_ShowerContinuous-v0')

---
# Semana 4

## Lab 3.6 · Hindsight Experience Replay (HER)
**Objetivo.** Ver el efecto de HER primero en un problema de juguete y después en un brazo robótico simulado en MuJoCo.

**Qué hacer**
1. **BitFlipping** (15 bits): DQN con y sin HER. Sin HER no hay señal de aprendizaje.
2. **FetchReach-v4**: SAC con y sin HER, 20k pasos, 3 semillas. Comparad `success_rate`.
3. Estrategias `final` / `future` / `episode` y `n_sampled_goal` = 1 / 4 / 8.
4. Ampliación (larga, en segundo plano): **FetchPush-v4**.

In [ ]:
lanzar('3.6_bitflip', 'python experiments_3/lab_3_6_her.py bitflip --seeds {s}', semillas=[0, 1, 2])
lanzar('3.6_fetch',   'python experiments_3/lab_3_6_her.py fetch --seeds {s}', semillas=[0, 1, 2])

In [ ]:
lanzar('3.6_estrategias', 'python experiments_3/lab_3_6_her.py fetch --her-only --strategy final,future,episode --seeds {s}', semillas=[0])
lanzar('3.6_k',           'python experiments_3/lab_3_6_her.py fetch --her-only --k 1,8 --seeds {s}', semillas=[0])   # k=4 ya está en la celda anterior
# Ampliación (horas): lanzar('3.6_push', 'python experiments_3/lab_3_6_her.py fetch --env FetchPush-v4 --her-only --steps 500000 --seeds 0')

In [ ]:
estado()

In [ ]:
grafica('logs/lab_3_6_bitflip', metric='success')
grafica('logs/lab_3_6_FetchReach-v4', metric='success')

In [ ]:
# El brazo en acción (GIF)
ejecutar('python experiments_3/lab_3_6_her.py show --model logs/lab_3_6_FetchReach-v4/her_future_k4/seed_0/best_model.zip --episodes 3 --gif logs/fetch.gif')
ver_gif('logs/fetch.gif')

## Lab 3.7 · Ajustar PPO con Optuna y defender el resultado
**Objetivo.** Aplicar el protocolo completo: buscar hiperparámetros, reentrenar con semillas **nuevas** y comparar honestamente.

**Qué hacer**
1. Búsqueda con 20 trials y poda (`MedianPruner`). Seguidla con optuna-dashboard (celda siguiente).
2. Comparar los mejores hiperparámetros con los de rl_zoo3.
3. Reentrenar la mejor configuración con 5 semillas que no se usaron al ajustar (100–104).
4. Gráfica final: DQN (3.1), A2C y PPO por defecto (3.4) y PPO ajustado.

Si va justo de tiempo: `--chunks 3`.

In [ ]:
lanzar('3.7_search', 'python experiments_3/lab_3_7_optuna.py search --trials 20')

In [ ]:
# optuna-dashboard dentro del cuaderno
from google.colab import output
subprocess.Popen('optuna-dashboard sqlite:///optuna/optuna.db --port 8081', shell=True)
time.sleep(5)
output.serve_kernel_port_as_iframe(8081, height=650)

In [ ]:
estado()

In [ ]:
lanzar('3.7_final', 'python experiments_3/lab_3_7_optuna.py final --seeds {s}', semillas=[100, 101, 102, 103, 104])

In [ ]:
grafica('logs/lab_3_7', 'logs/lab_3_4', 'logs/lab_3_1',
        configs='ppo_tuned,ppo_default,a2c_default,LunarLander-v3_slides', out='logs/comparacion_final.png')

---
## Problemas frecuentes en Colab
- **Se ha reiniciado el entorno**: volved a ejecutar la sección 0 completa. Los resultados siguen en Drive; `experiments_3/` no se sobrescribe.
- **`estado()` dice ERROR**: mirad el final de `logs/procesos/<nombre>.log`.
- **Va lento**: comprobad `nproc`; con pocos núcleos, la cola alarga los tiempos. Reducid semillas o pasos para probar.
- **MuJoCo no dibuja**: comprobad que `MUJOCO_GL=egl` se ha fijado en 0.3 antes de importar nada.
- **Box2D no instala**: falta `swig` (primera línea de 0.3).